# 02 — Завантаження VNP46A4 і агрегація нічного освітлення на рівні країн

Ноутбук формує панель «країна — рік» за річними композитами NASA Black Marble **VNP46A4** (колекція 002) для 35 європейських країн за 2012–2025 роки.

**Етапи:**
1. Налаштування середовища, шляхів і параметрів.
2. Область дослідження: полігони країн, геодезична площа, мапа «країна — тайл».
3. Маніфест файлів VNP46A4 через LAADS API і перевірка його повноти.
4. Завантаження тайла з перевіркою цілісності та читання шарів HDF5.
5. Виділення пікселів країни і агрегація статистик.
6. Обробка одного року.
7. Обробка всіх років і збирання панелі.
8. Контроль якості панелі.
9. Порівняння ознак нічного освітлення з ВВП.

**Вихідні файли** (у `data/processed/ntl/`): `vnp46a4_country_ntl_{year}.csv`, `ntl_panel_2012_2025.csv`, `vnp46a4_processing_summary_2012_2025.csv`.

## 1. Налаштування середовища та параметрів

Підключення Google Drive, імпорти, шляхи проєкту і параметри обробки:

- `VALID_SURFACE_CLASSES` — класи `Land_Water_Mask`, що вважаються суходолом: 0 (Land & Desert), 1 (Land no Desert), 5 (Coastal);
- `LIT_RADIANCE_THRESHOLD` і `LIT_THRESHOLDS` — пороги радіації (nW/(cm²·sr)) для освітленої площі;
- `REMOTE_TILES` — тайли віддалених територій (заморські департаменти Франції, Азорські й Канарські острови, Мадейра), які виключаються у варіанті `*_mainland`;
- `REQUIRED_COLUMNS` — колонки, без яких річний CSV вважається застарілим і перераховується;
- `REFRESH_MANIFEST` — чи збирати маніфест заново (посилання LAADS можуть застаріти, а запитів лише 14, тому за замовчуванням так).

Токен NASA Earthdata береться з Colab Secrets (`NTLDATA_TOKEN`). Він діє 60 днів.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
from collections import defaultdict
from functools import lru_cache
from pathlib import Path
import gc
import hashlib
import re
import time

import geopandas as gpd
import h5py
import numpy as np
import pandas as pd
import requests
from google.colab import userdata
from pyproj import Geod
from rasterio.features import geometry_mask
from rasterio.transform import from_bounds
from rasterio.windows import Window, transform as window_transform
from shapely.geometry import box

# --- Шляхи ---
PROJECT_ROOT = Path("/content/drive/MyDrive/GDP_NTL_Project")
METADATA_DIR = PROJECT_ROOT / "data" / "metadata"
NTL_PROCESSED_DIR = PROJECT_ROOT / "data" / "processed" / "ntl"
TEMP_DIR = Path("/content/tmp_vnp46a4")

STUDY_AREA_PATH = METADATA_DIR / "study_area.geojson"
COUNTRY_TILE_PATH = METADATA_DIR / "country_tile_map.csv"
MANIFEST_PATH = METADATA_DIR / "vnp46a4_manifest_2012_2025.csv"
SUMMARY_PATH = NTL_PROCESSED_DIR / "vnp46a4_processing_summary_2012_2025.csv"
PANEL_PATH = NTL_PROCESSED_DIR / "ntl_panel_2012_2025.csv"
GDP_PATH = PROJECT_ROOT / "data" / "economic" / "eurostat_nama_10_gdp_2012_2025.csv"

for path in [METADATA_DIR, NTL_PROCESSED_DIR, TEMP_DIR]:
    path.mkdir(parents=True, exist_ok=True)

# --- Продукт і API ---
API_URL = "https://ladsweb.modaps.eosdis.nasa.gov/api/v2/content/details"
PRODUCT = "VNP46A4"
YEARS = range(2012, 2026)
REFERENCE_YEAR = 2024
REFRESH_MANIFEST = True
DATA_FIELDS = "HDFEOS/GRIDS/VIIRS_Grid_DNB_2d/Data Fields/"
HDF5_SIGNATURE = b"\x89HDF\r\n\x1a\n"
FILENAME_RE = re.compile(r"\.A(\d{4})\d{3}\.(h\d{2}v\d{2})\.")

# --- Класи поверхні та пороги ---
VALID_SURFACE_CLASSES = [0, 1, 5]       # Land & Desert, Land no Desert, Coastal
WATER_CLASSES = [2, 3]                  # Inland Water, Sea Water
KNOWN_SURFACE_CLASSES = VALID_SURFACE_CLASSES + WATER_CLASSES
LIT_RADIANCE_THRESHOLD = 0.5            # nW/(cm² sr)
LIT_THRESHOLDS = {"t1": 1, "t2": 2, "t5": 5}

# --- Агрегація ---
REMOTE_TILES = {"h11v07", "h12v08", "h22v10", "h23v11",
                "h14v05", "h15v05", "h16v05", "h16v06"}
QA_KEYS = ["water_km2", "surface_fill_km2", "invalid_radiance_km2",
           "qa0_km2", "qa1_km2", "qa2_km2", "other_km2", "total_mask_km2"]
MAINLAND_KEYS = ["valid_ntl_area_km2", "ntl_sum_area_weighted",
                 "ntl_mean_area_weighted", "lit_area_km2", "ntl_sum_cap50"]
REQUIRED_COLUMNS = ["ntl_sum_cap50"]

GEOD = Geod(ellps="WGS84")

NASA_TOKEN = userdata.get("NTLDATA_TOKEN")
if not NASA_TOKEN:
    raise ValueError("NTLDATA_TOKEN not found in Colab Secrets.")

### Допоміжні функції загального призначення

- `load_or_build_csv` — читає CSV з Drive або будує його функцією-будівником і зберігає (використовується для мапи тайлів і маніфесту);
- `geodesic_area_km2` — геодезична площа геометрії на еліпсоїді WGS84;
- `tile_bounds` — межі тайла 10°×10° за назвою `hXXvYY`;
- `year_csv_path` — шлях до річного CSV.

In [ ]:
def load_or_build_csv(path, builder, rebuild=False):
    """Читає CSV, якщо він є; інакше (або при rebuild=True) будує і зберігає."""
    if path.exists() and not rebuild:
        return pd.read_csv(path)
    df = builder()
    df.to_csv(path, index=False)
    return df


def geodesic_area_km2(geometry):
    area_m2, _ = GEOD.geometry_area_perimeter(geometry)
    return abs(area_m2) / 1_000_000


def tile_bounds(tile):
    """(west, south, east, north) тайла hXXvYY фіксованої сітки 10°×10°."""
    h, v = int(tile[1:3]), int(tile[4:6])
    west, north = -180 + h * 10, 90 - v * 10
    return west, north - 10, west + 10, north


def year_csv_path(year):
    return NTL_PROCESSED_DIR / f"vnp46a4_country_ntl_{year}.csv"

## 2. Область дослідження і потрібні тайли

Завантаження полігонів 35 країн і розрахунок їхньої геодезичної площі (WGS84). Площа використовується як знаменник для часток покриття та розбивки за якістю.

`COUNTRY_GEOMS` — словник «код країни → геометрія», щоб не шукати геометрію в таблиці при кожній обробці тайла.

In [ ]:
study_area = gpd.read_file(STUDY_AREA_PATH)
study_area["country_area_km2"] = study_area.geometry.apply(geodesic_area_km2)
COUNTRY_GEOMS = dict(zip(study_area["CNTR_ID"], study_area.geometry))

print("Countries:", study_area["CNTR_ID"].nunique(), "| CRS:", study_area.crs)

Countries: 35 | CRS: EPSG:4326


### Мапа «країна — тайл»

Тайли VNP46A4 утворюють фіксовану сітку 10°×10° (h00–h35, v00–v17), тому перетин країн із тайлами рахується локально, без запитів до API. Результат кешується в `country_tile_map.csv`.

`TILE_COUNTRIES` — словник «тайл → список країн», який використовується під час обробки.

In [ ]:
def build_country_tile_map():
    grid = gpd.GeoDataFrame(
        [{"tile": f"h{h:02d}v{v:02d}", "geometry": box(*tile_bounds(f"h{h:02d}v{v:02d}"))}
         for h in range(36) for v in range(18)],
        crs="EPSG:4326",
    )
    return (
        gpd.sjoin(study_area[["CNTR_ID", "NAME_ENGL", "geometry"]], grid,
                  how="inner", predicate="intersects")
        [["CNTR_ID", "NAME_ENGL", "tile"]]
        .drop_duplicates()
        .sort_values(["CNTR_ID", "tile"])
        .reset_index(drop=True)
    )


country_tile_map = load_or_build_csv(COUNTRY_TILE_PATH, build_country_tile_map)
TILE_COUNTRIES = country_tile_map.groupby("tile")["CNTR_ID"].apply(list).to_dict()
required_tiles = sorted(TILE_COUNTRIES)
print("Required tiles:", len(required_tiles))   # очікується 31

Required tiles: 31


## 3. Маніфест даних

`build_vnp46a4_manifest` запитує LAADS API-V2 по одному року (з урахуванням пагінації) і залишає лише файли потрібних тайлів: назву, розмір, посилання, MD5 і статус.

`validate_manifest` одним зведенням перевіряє маніфест: кожна пара «рік × тайл» присутня рівно один раз, усі файли онлайн, усі належать одній колекції. Якщо бракує файлів або є дублікати, виконання зупиняється.

In [ ]:
def _api_get(params):
    response = requests.get(API_URL, params=params, timeout=120)
    response.raise_for_status()
    return response.json()


def build_vnp46a4_manifest(years=YEARS, tiles=None):
    tiles = set(tiles or required_tiles)
    records = []
    for year in years:
        print(f"Checking {year}...")
        params = {"products": PRODUCT, "temporalRanges": f"{year}-01-01..{year}-12-31"}
        first = _api_get(params)
        pages = [first] + [_api_get({**params, "page": p})
                           for p in range(2, first.get("n_pages", 1) + 1)]
        for page in pages:
            for f in page.get("content", []):
                match = FILENAME_RE.search(f.get("name", ""))
                if match and int(match.group(1)) == year and match.group(2) in tiles:
                    records.append({
                        "year": year,
                        "tile": match.group(2),
                        "filename": f.get("name"),
                        "size_bytes": f.get("size"),
                        "download_url": f.get("downloadsLink"),
                        "md5": f.get("md5sum"),
                        "status": f.get("status"),
                    })
    return pd.DataFrame(records).sort_values(["year", "tile"]).reset_index(drop=True)


def validate_manifest(manifest):
    expected = {(y, t) for y in YEARS for t in required_tiles}
    found = set(zip(manifest["year"], manifest["tile"]))
    missing = sorted(expected - found)
    n_duplicates = int(manifest.duplicated(["year", "tile"]).sum())
    n_offline = int((manifest["status"] != "Online").sum())
    collections = sorted(manifest["filename"].str.extract(r"\.(\d{3})\.\d{13}\.h5$")[0].dropna().unique())

    print(f"Files: {len(manifest)} / {len(expected)} | missing: {len(missing)} | "
          f"duplicates: {n_duplicates} | not online: {n_offline} | collections: {collections} | "
          f"volume: {manifest['size_bytes'].sum() / 1e9:.1f} GB")
    assert not missing, f"Missing year × tile: {missing}"
    assert n_duplicates == 0, "Duplicate year × tile combinations in manifest"


availability_ntl = load_or_build_csv(MANIFEST_PATH, build_vnp46a4_manifest, rebuild=REFRESH_MANIFEST)
validate_manifest(availability_ntl)

Checking 2012...
Checking 2013...
Checking 2014...
Checking 2015...
Checking 2016...
Checking 2017...
Checking 2018...
Checking 2019...
Checking 2020...
Checking 2021...
Checking 2022...
Checking 2023...
Checking 2024...
Checking 2025...
Files: 434 / 434 | missing: 0 | duplicates: 0 | not online: 0 | collections: ['002'] | volume: 44.6 GB


## 4. Завантаження і читання тайлів

- `download_vnp46a4_file` — потокове завантаження з одночасним розрахунком MD5 (файл не перечитується повторно), перевіркою сигнатури HDF5 і розміру. Мережеві збої та розбіжності розміру/MD5 повторюються до трьох разів. Якщо сервер повернув не HDF5 (зазвичай сторінку входу Earthdata через прострочений токен), повтор не виконується;
- `read_vnp46a4_tile` — читає радіацію, прапорець якості, кількість спостережень і маску «суходіл-вода», одразу рахує маски валідної радіації та суходолу і площу пікселів кожного рядка;
- `load_tile` — завантажує тайл, читає його в пам'ять і видаляє файл з диска;
- `get_file_info` — рядок маніфесту для пари «рік × тайл».

Площа пікселя залежить лише від широти, тому вона рахується один раз на рядок тайла і кешується (`lru_cache`).

In [ ]:
class NotHDF5Error(ValueError):
    """Сервер повернув не HDF5 (ймовірно, прострочений токен)."""


def download_vnp46a4_file(file_info, retries=3, chunk_size=1024 * 1024):
    local_path = TEMP_DIR / file_info["filename"]
    headers = {"Authorization": f"Bearer {NASA_TOKEN}"}

    for attempt in range(1, retries + 1):
        try:
            md5, size, signature = hashlib.md5(), 0, b""
            with requests.get(file_info["download_url"], headers=headers,
                              stream=True, timeout=(30, 300)) as response, \
                 open(local_path, "wb") as f:
                response.raise_for_status()
                for chunk in response.iter_content(chunk_size=chunk_size):
                    if size == 0:
                        signature = chunk[:8]
                    f.write(chunk)
                    md5.update(chunk)
                    size += len(chunk)

            if signature != HDF5_SIGNATURE:
                raise NotHDF5Error(f"{file_info['filename']}: not an HDF5 file "
                                   "(check that NTLDATA_TOKEN is still valid).")
            if size != int(file_info["size_bytes"]):
                raise ValueError(f"File size mismatch: {file_info['filename']}")
            if md5.hexdigest().lower() != str(file_info["md5"]).lower():
                raise ValueError(f"MD5 mismatch: {file_info['filename']}")
            return local_path

        except (requests.RequestException, ValueError) as exc:
            local_path.unlink(missing_ok=True)
            if isinstance(exc, NotHDF5Error) or attempt == retries:
                raise
            print(f"  attempt {attempt}/{retries} failed: {exc}")
            time.sleep(10 * attempt)


@lru_cache(maxsize=None)
def _row_areas_km2(north_top, pixel_width, pixel_height, height):
    """Площа пікселя (км²) для кожного рядка тайла; від довготи не залежить."""
    areas = np.empty(height, dtype=np.float64)
    for row in range(height):
        north = north_top - row * pixel_height
        south = north - pixel_height
        area_m2, _ = GEOD.polygon_area_perimeter(
            [0.0, pixel_width, pixel_width, 0.0], [north, north, south, south]
        )
        areas[row] = abs(area_m2) / 1_000_000
    return areas


def pixel_area_by_row(transform, height):
    return _row_areas_km2(round(transform.f, 6), round(transform.a, 9),
                          round(abs(transform.e), 9), height)


def read_vnp46a4_tile(file_path, tile):
    with h5py.File(file_path, "r") as h5:
        def read(name, default_fill):
            ds = h5[DATA_FIELDS + name]
            return ds[:], ds.attrs.get("_FillValue", default_fill)

        radiance, rad_fill = read("NearNadir_Composite_Snow_Free", -999.9)
        quality, _ = read("NearNadir_Composite_Snow_Free_Quality", 255)
        n_obs, nobs_fill = read("NearNadir_Composite_Snow_Free_Num", 65535)
        surface, _ = read("Land_Water_Mask", 255)

    radiance = radiance.astype(np.float32)
    n_obs = n_obs.astype(np.float32)
    n_obs[n_obs == nobs_fill] = np.nan          # n_obs без маскування за якістю/поверхнею

    west, south, east, north = tile_bounds(tile)
    height, width = radiance.shape
    transform = from_bounds(west, south, east, north, width, height)

    return {
        "tile": tile,
        "radiance": radiance,
        "quality": quality,
        "n_obs": n_obs,
        "surface": surface,
        "valid_radiance": ~np.isclose(radiance, rad_fill) & (radiance >= 0),
        "valid_surface": np.isin(surface, VALID_SURFACE_CLASSES),
        "transform": transform,
        "bounds": (west, south, east, north),
        "row_area": pixel_area_by_row(transform, height),
    }


def get_file_info(year, tile):
    return availability_ntl.loc[(availability_ntl["year"] == year)
                                & (availability_ntl["tile"] == tile)].iloc[0]


def load_tile(file_info):
    """Завантажує тайл, читає його в пам'ять і одразу видаляє файл."""
    path = download_vnp46a4_file(file_info)
    try:
        return read_vnp46a4_tile(path, file_info["tile"])
    finally:
        path.unlink(missing_ok=True)

## 5. Виділення пікселів країни та агрегація

**`country_window`** обрізає геометрію країни межами тайла і растеризує її лише у вікні, що покриває обрізану геометрію, а не на всьому тайлі 2400×2400. Для маленьких країн це в десятки разів менше роботи. Піксель належить країні, якщо його центр лежить усередині полігона (`all_touched=False`).

**`extract_country_tile`** за одну растеризацію повертає три результати:
- `qa0` — пікселі суходолу з валідною радіацією і `quality == 0` (основний варіант);
- `qa01` — те саме з `quality ∈ {0, 1}` (альтернативний варіант);
- `qa` — взаємовиключна розбивка площі маски країни: вода, fill маски поверхні, невалідна радіація, QA0/QA1/QA2, інше.

**`aggregate_country_pixels`** рахує статистики за пікселями всіх тайлів країни:
- розподіл радіації без зважування: середнє, std, перцентилі, IQR, CV;
- суми, зважені за площею: `ntl_sum_area_weighted`, обрізані на 50 і 100 (`cap50`, `cap100`) та `log1p`;
- частку суми від пікселів з радіацією ≥ 100;
- освітлену площу за порогами 0.5, 1, 2 і 5;
- середню кількість спостережень VIIRS.

In [ ]:
def country_window(tile_data, geometry):
    """((row_slice, col_slice), mask) для країни в тайлі; None, якщо перетину немає."""
    west, south, east, north = tile_data["bounds"]
    clipped = geometry.intersection(box(west, south, east, north))
    if clipped.is_empty:
        return None

    t = tile_data["transform"]
    height, width = tile_data["radiance"].shape
    pixel_w, pixel_h = t.a, -t.e
    minx, miny, maxx, maxy = clipped.bounds
    col0 = max(int(np.floor((minx - west) / pixel_w)), 0)
    col1 = min(int(np.ceil((maxx - west) / pixel_w)), width)
    row0 = max(int(np.floor((north - maxy) / pixel_h)), 0)
    row1 = min(int(np.ceil((north - miny) / pixel_h)), height)
    if col1 <= col0 or row1 <= row0:
        return None

    window = Window(col0, row0, col1 - col0, row1 - row0)
    mask = geometry_mask([clipped], out_shape=(row1 - row0, col1 - col0),
                         transform=window_transform(window, t),
                         invert=True, all_touched=False)
    return (slice(row0, row1), slice(col0, col1)), mask


def extract_country_tile(tile_data, geometry):
    """Пікселі QA0, QA0+1 і розбивка площі за якістю за одну растеризацію."""
    found = country_window(tile_data, geometry)
    if found is None:
        return None
    (rows, cols), mask = found

    radiance = tile_data["radiance"][rows, cols]
    quality = tile_data["quality"][rows, cols]
    surface = tile_data["surface"][rows, cols]
    n_obs = tile_data["n_obs"][rows, cols]
    valid_radiance = tile_data["valid_radiance"][rows, cols]
    valid_surface = tile_data["valid_surface"][rows, cols]
    row_area = tile_data["row_area"][rows]

    land_ok = mask & valid_surface & valid_radiance

    def pixels(selection):
        if not selection.any():
            return None
        r, c = np.nonzero(selection)
        return {"radiance": radiance[r, c].astype(np.float64),
                "area_km2": row_area[r],
                "n_obs": n_obs[r, c]}

    qa_masks = {
        "water_km2": mask & np.isin(surface, WATER_CLASSES),
        "surface_fill_km2": mask & ~np.isin(surface, KNOWN_SURFACE_CLASSES),
        "invalid_radiance_km2": mask & valid_surface & ~valid_radiance,
        "qa0_km2": land_ok & (quality == 0),
        "qa1_km2": land_ok & (quality == 1),
        "qa2_km2": land_ok & (quality == 2),
        "other_km2": land_ok & ~np.isin(quality, [0, 1, 2]),
        "total_mask_km2": mask,
    }
    return {
        "qa0": pixels(qa_masks["qa0_km2"]),
        "qa01": pixels(land_ok & np.isin(quality, [0, 1])),
        "qa": {name: float(m.sum(axis=1) @ row_area) for name, m in qa_masks.items()},
    }


def aggregate_country_pixels(pixel_parts, include_nobs=True):
    parts = [p for p in pixel_parts if p is not None]
    if not parts:
        return None

    radiance = np.concatenate([p["radiance"] for p in parts])
    area = np.concatenate([p["area_km2"] for p in parts])
    weighted = radiance * area

    valid_area = area.sum()
    ntl_sum = weighted.sum()
    ntl_mean, ntl_std = radiance.mean(), radiance.std()
    p25, p50, p75, p90, p95 = np.percentile(radiance, [25, 50, 75, 90, 95])
    lit = radiance >= LIT_RADIANCE_THRESHOLD

    out = {
        "n_valid_pixels": int(radiance.size),
        "valid_ntl_area_km2": float(valid_area),
        # розподіл (без зважування за площею)
        "ntl_mean": float(ntl_mean),
        "ntl_median": float(p50),
        "ntl_std": float(ntl_std),
        "ntl_p25": float(p25),
        "ntl_p75": float(p75),
        "ntl_p90": float(p90),
        "ntl_p95": float(p95),
        "ntl_iqr": float(p75 - p25),
        "ntl_cv": float(ntl_std / ntl_mean) if ntl_mean > 0 else np.nan,
        # зважені за площею
        "ntl_mean_area_weighted": float(ntl_sum / valid_area),
        "ntl_sum_area_weighted": float(ntl_sum),
        "ntl_sum_cap50": float(np.minimum(radiance, 50) @ area),
        "ntl_sum_cap100": float(np.minimum(radiance, 100) @ area),
        "ntl_sum_log1p": float(np.log1p(radiance) @ area),
        "ntl_share_ge100": float(weighted[radiance >= 100].sum() / ntl_sum) if ntl_sum > 0 else np.nan,
        # освітлена територія
        "lit_area_km2": float(area[lit].sum()),
        "lit_area_share_valid": float(area[lit].sum() / valid_area),
        "ntl_positive_mean": float(radiance[lit].mean()) if lit.any() else np.nan,
        **{f"lit_area_km2_{name}": float(area[radiance >= thr].sum())
           for name, thr in LIT_THRESHOLDS.items()},
    }

    if include_nobs:
        n_obs = np.concatenate([p["n_obs"] for p in parts])
        out["mean_n_observations"] = float(np.nanmean(n_obs)) if (~np.isnan(n_obs)).any() else np.nan

    return out

## 6. Обробка одного року

**`build_country_row`** збирає рядок країни з результатів по всіх її тайлах:
- основні показники (`quality == 0`) і частка покриття `ntl_coverage_share`;
- розбивка площі за якістю в км² і як частка площі полігона (`*_share`);
- варіант `*_qa01` (`quality ∈ {0, 1}`);
- варіант `*_mainland` без віддалених тайлів (лише для країн, що їх мають).

**`process_ntl_year`** по черзі завантажує всі тайли року, для кожного тайла обробляє країни, що його перетинають, і зберігає `vnp46a4_country_ntl_{year}.csv`. Наявний файл року просто читається, якщо не задано `overwrite=True`.

In [ ]:
def build_country_row(country, tile_results, year):
    """tile_results — список (tile, результат extract_country_tile або None)."""
    code, area = country["CNTR_ID"], country["country_area_km2"]
    results = [(t, r) for t, r in tile_results if r is not None]

    main = aggregate_country_pixels([r["qa0"] for _, r in results], include_nobs=True)
    if main is None:
        print(f"WARNING: no valid pixels for {code} in {year}")
        return None

    qa_sum = {k: sum(r["qa"][k] for _, r in results) for k in QA_KEYS}
    row = {
        "CNTR_ID": code, "NAME_ENGL": country["NAME_ENGL"], "year": year,
        "country_area_km2": area, **main,
        "ntl_coverage_share": main["valid_ntl_area_km2"] / area,
        **qa_sum,
        **{k.replace("_km2", "_share"): qa_sum[k] / area for k in QA_KEYS if k != "total_mask_km2"},
    }

    alt = aggregate_country_pixels([r["qa01"] for _, r in results], include_nobs=False)
    if alt is not None:
        row.update({
            "valid_ntl_area_km2_qa01": alt["valid_ntl_area_km2"],
            "ntl_mean_area_weighted_qa01": alt["ntl_mean_area_weighted"],
            "ntl_sum_area_weighted_qa01": alt["ntl_sum_area_weighted"],
            "ntl_coverage_share_qa01": alt["valid_ntl_area_km2"] / area,
        })

    if any(t in REMOTE_TILES for t, _ in tile_results):
        mainland = aggregate_country_pixels(
            [r["qa0"] for t, r in results if t not in REMOTE_TILES], include_nobs=False
        )
        if mainland is not None:
            row.update({f"{k}_mainland": mainland[k] for k in MAINLAND_KEYS})
    return row


def process_ntl_year(year, overwrite=False):
    output_path = year_csv_path(year)
    if output_path.exists() and not overwrite:
        print("Already exists, skipping:", output_path.name)
        return pd.read_csv(output_path)

    files = availability_ntl.loc[availability_ntl["year"] == year].sort_values("tile")
    if files.empty:
        raise ValueError(f"No VNP46A4 files found for {year}.")

    tile_results = defaultdict(list)
    for i, (_, file_info) in enumerate(files.iterrows(), start=1):
        tile = file_info["tile"]
        print(f"[{i}/{len(files)}] {year} {tile}")
        tile_data = load_tile(file_info)
        for code in TILE_COUNTRIES[tile]:
            tile_results[code].append((tile, extract_country_tile(tile_data, COUNTRY_GEOMS[code])))
        del tile_data
        gc.collect()

    rows = [build_country_row(country, tile_results[country["CNTR_ID"]], year)
            for _, country in study_area.iterrows()]
    result = pd.DataFrame([r for r in rows if r is not None]).sort_values("CNTR_ID").reset_index(drop=True)
    assert result["CNTR_ID"].is_unique

    result.to_csv(output_path, index=False)
    print("Saved:", output_path.name)
    return result

## 7. Обробка всіх років і збирання панелі

- `year_needs_recalc` — рік перераховується, якщо його CSV немає або в ньому бракує колонок із `REQUIRED_COLUMNS`;
- `run_all_years` — обробляє роки по черзі. Помилка в одному році не зупиняє решту, а потрапляє у зведення `vnp46a4_processing_summary_2012_2025.csv` (статус, мінімальне покриття QA0 і QA0+1, максимальні частки QA1/QA2, час);
- `build_ntl_panel` — об'єднує всі річні CSV у панель `ntl_panel_2012_2025.csv` і перевіряє, що пари «країна × рік» не дублюються.

In [ ]:
def year_needs_recalc(year):
    path = year_csv_path(year)
    return not path.exists() or not set(REQUIRED_COLUMNS) <= set(pd.read_csv(path, nrows=0).columns)


def run_all_years(years=YEARS):
    summary = []
    for year in years:
        t0 = time.time()
        record = {"year": year}
        try:
            res = process_ntl_year(year, overwrite=year_needs_recalc(year))
            record.update({
                "status": "success",
                "n_countries": len(res),
                "min_qa0_coverage": res["ntl_coverage_share"].min(),
                "min_qa01_coverage": res["ntl_coverage_share_qa01"].min(),
                "max_qa1_share": res["qa1_share"].max(),
                "max_qa2_share": res["qa2_share"].max(),
                "error": None,
            })
        except Exception as exc:
            print(f"Error in {year}: {exc}")
            record.update({"status": "error", "error": str(exc)})
            gc.collect()
        record["seconds"] = round(time.time() - t0)
        summary.append(record)

    summary = pd.DataFrame(summary)
    summary.to_csv(SUMMARY_PATH, index=False)
    print(f"Successful years: {(summary['status'] == 'success').sum()} / {len(summary)}")
    return summary


def build_ntl_panel():
    files = sorted(NTL_PROCESSED_DIR.glob("vnp46a4_country_ntl_*.csv"))
    panel = (pd.concat((pd.read_csv(f) for f in files), ignore_index=True)
               .sort_values(["CNTR_ID", "year"]).reset_index(drop=True))
    assert not panel.duplicated(["CNTR_ID", "year"]).any(), "Duplicate country-year rows"

    expected = len(study_area) * len(YEARS)
    print(f"Files: {len(files)} | rows: {len(panel)} | expected: {expected}")
    if len(panel) != expected:
        print("WARNING: panel is incomplete")
    panel.to_csv(PANEL_PATH, index=False)
    return panel

In [ ]:
processing_summary = run_all_years()
display(processing_summary)

ntl_panel = build_ntl_panel()

[1/31] 2012 h11v07
[2/31] 2012 h12v08
[3/31] 2012 h14v05
[4/31] 2012 h15v02
[5/31] 2012 h15v05
[6/31] 2012 h16v02
[7/31] 2012 h16v03
[8/31] 2012 h16v05
[9/31] 2012 h16v06
[10/31] 2012 h17v03
[11/31] 2012 h17v04
[12/31] 2012 h17v05
[13/31] 2012 h18v02
[14/31] 2012 h18v03
[15/31] 2012 h18v04
[16/31] 2012 h18v05
[17/31] 2012 h19v01
[18/31] 2012 h19v02
[19/31] 2012 h19v03
[20/31] 2012 h19v04
[21/31] 2012 h19v05
[22/31] 2012 h20v01
[23/31] 2012 h20v02
[24/31] 2012 h20v03
[25/31] 2012 h20v04
[26/31] 2012 h20v05
[27/31] 2012 h21v01
[28/31] 2012 h21v02
[29/31] 2012 h21v05
[30/31] 2012 h22v10
[31/31] 2012 h23v11
Saved: vnp46a4_country_ntl_2012.csv
[1/31] 2013 h11v07
[2/31] 2013 h12v08
[3/31] 2013 h14v05
[4/31] 2013 h15v02
[5/31] 2013 h15v05
[6/31] 2013 h16v02
[7/31] 2013 h16v03
[8/31] 2013 h16v05
[9/31] 2013 h16v06
[10/31] 2013 h17v03
[11/31] 2013 h17v04
[12/31] 2013 h17v05
[13/31] 2013 h18v02
[14/31] 2013 h18v03
[15/31] 2013 h18v04
[16/31] 2013 h18v05
[17/31] 2013 h19v01
[18/31] 2013 h19v02
[1

,year,status,n_countries,min_qa0_coverage,min_qa01_coverage,max_qa1_share,max_qa2_share,error,seconds
0,2012,success,35,0.726850,0.946259,0.263556,0.010998,None,148
1,2013,success,35,0.827170,0.946209,0.163748,0.013012,None,120
2,2014,success,35,0.791338,0.946238,0.186703,0.014031,None,136
3,2015,success,35,0.727623,0.946230,0.251185,0.013321,None,170
4,2016,success,35,0.882383,0.946286,0.065568,0.011827,None,121
5,2017,success,35,0.816414,0.946209,0.174930,0.007519,None,145
6,2018,success,35,0.863779,0.946269,0.082491,0.010673,None,125
7,2019,success,35,0.765368,0.946255,0.224606,0.010827,None,205
8,2020,success,35,0.829813,0.946255,0.147772,0.014513,None,136
9,2021,success,35,0.769387,0.946301,0.209206,0.014528,None,108


Files: 14 | rows: 490 | expected: 490


## 8. Контроль якості панелі

**`panel_qa_report`** перевіряє геометрію і стабільність фільтра якості:
1. повнота розбивки: сума категорій площі дорівнює площі маски країни (допуск 0.01 %);
2. маска відтворює площу полігона (допуск 2 %);
3. `qa0_share_land` — частка суходолу, що пройшла фільтр `quality == 0`, і її розмах між роками. Великий розмах означає, що зміна покриття може створювати штучний тренд у сумі світла.

**`flag_large_changes`** показує медіанну річну зміну показника по країнах (спільні для всіх країн стрибки вказують на зміни в продукті, а не в економіці) і пари «країна × рік» зі зміною понад поріг.

In [ ]:
QA_PARTITION_COLS = [k for k in QA_KEYS if k != "total_mask_km2"]
QA_LAND_COLS = ["invalid_radiance_km2", "qa0_km2", "qa1_km2", "qa2_km2", "other_km2"]


def panel_qa_report(panel, partition_tol_pct=0.01, polygon_tol_pct=2.0):
    qa = panel[["CNTR_ID", "year", "country_area_km2", "total_mask_km2"] + QA_PARTITION_COLS].copy()
    qa["partition_err_pct"] = (qa[QA_PARTITION_COLS].sum(axis=1) / qa["total_mask_km2"] - 1) * 100
    qa["mask_vs_polygon_pct"] = (qa["total_mask_km2"] / qa["country_area_km2"] - 1) * 100
    qa["qa0_share_land"] = qa["qa0_km2"] / qa[QA_LAND_COLS].sum(axis=1)

    print(qa[["partition_err_pct", "mask_vs_polygon_pct"]].describe().round(3))
    bad = qa[(qa["partition_err_pct"].abs() > partition_tol_pct)
             | (qa["mask_vs_polygon_pct"].abs() > polygon_tol_pct)]
    print("Problem country-years:", len(bad))
    if not bad.empty:
        display(bad.sort_values("mask_vs_polygon_pct"))

    share = qa.pivot(index="CNTR_ID", columns="year", values="qa0_share_land")
    print("\nRange of qa0_share_land across years (top 10):")
    print((share.max(axis=1) - share.min(axis=1)).round(3).sort_values(ascending=False).head(10))
    return qa


def flag_large_changes(panel, col="ntl_sum_area_weighted", threshold_pct=15):
    change = panel.pivot(index="CNTR_ID", columns="year", values=col).pct_change(axis=1) * 100
    print(f"Median year-on-year change of {col} across countries, %:")
    print(change.median().round(1).to_string())
    big = change.stack().rename("pct").reset_index()
    return big.loc[big["pct"].abs() > threshold_pct].sort_values("pct").round(1).reset_index(drop=True)


qa_country = panel_qa_report(ntl_panel)
display(flag_large_changes(ntl_panel))

       partition_err_pct  mask_vs_polygon_pct
count              490.0              490.000
mean                 0.0                0.005
std                  0.0                0.025
min                 -0.0               -0.014
25%                 -0.0               -0.002
50%                  0.0               -0.000
75%                  0.0                0.005
max                  0.0                0.138
Problem country-years: 0

Range of qa0_share_land across years (top 10):
CNTR_ID
NO    0.289
IS    0.255
FI    0.217
SE    0.204
BE    0.062
NL    0.057
EE    0.031
DK    0.018
PL    0.017
DE    0.017
dtype: float64
Median year-on-year change of ntl_sum_area_weighted across countries, %:
year
2012    NaN
2013   -3.1
2014   -1.2
2015   -0.5
2016    0.7
2017    2.1
2018    1.9
2019   -0.3
2020   -2.0
2021    6.9
2022    6.0
2023   -0.8
2024   -0.0
2025    6.7


,CNTR_ID,year,pct
0,IS,2015,-54.0
1,IS,2018,-50.7
2,IS,2013,-34.6
3,NL,2022,-32.4
4,IS,2016,-32.0
5,FR,2023,-25.2
6,IS,2017,-23.5
7,FI,2017,-22.0
8,BE,2023,-21.6
9,NO,2020,-19.6


## 9. Порівняння ознак нічного освітлення з ВВП

- `load_gdp` — читає таблицю Eurostat `nama_10_gdp` і переводить її в довгий формат «країна — рік — ВВП»;
- `ntl_gdp_correlations` — для кожної ознаки рахує кореляцію логарифмів з логарифмом ВВП:
  - **між країнами** в опорному році (`REFERENCE_YEAR`);
  - **всередині країн** (після віднімання середнього по країні, тобто за динамікою);
  - кількість країн із від'ємною кореляцією в часі.

Перша таблиця — зведення за ознаками, друга — кореляції в часі для окремих країн.

In [ ]:
NTL_FEATURES = ["ntl_sum_area_weighted", "ntl_sum_cap50", "ntl_sum_cap100", "ntl_sum_log1p",
                "lit_area_km2_t1", "lit_area_km2_t2", "lit_area_km2_t5"]


def load_gdp(path=GDP_PATH):
    raw = pd.read_csv(path)
    geo_col = next(c for c in raw.columns if str(c).startswith("geo"))
    year_cols = [c for c in raw.columns if str(c).isdigit()]
    gdp = (raw.rename(columns={geo_col: "CNTR_ID"})[["CNTR_ID"] + year_cols]
              .melt(id_vars="CNTR_ID", var_name="year", value_name="gdp"))
    gdp["year"] = gdp["year"].astype(int)
    gdp["gdp"] = pd.to_numeric(gdp["gdp"], errors="coerce")
    return gdp.dropna(subset=["gdp"])


def ntl_gdp_correlations(data, features=NTL_FEATURES, year=REFERENCE_YEAR):
    summary, per_country = [], {}
    for col in features:
        tmp = pd.DataFrame({
            "CNTR_ID": data["CNTR_ID"], "year": data["year"],
            "ln_x": np.log(data[col].where(data[col] > 0)),
            "ln_gdp": np.log(data["gdp"]),
        }).dropna()

        between = tmp.loc[tmp["year"] == year, "ln_x"].corr(tmp.loc[tmp["year"] == year, "ln_gdp"])
        demeaned = tmp[["ln_x", "ln_gdp"]] - tmp.groupby("CNTR_ID")[["ln_x", "ln_gdp"]].transform("mean")
        within = demeaned["ln_x"].corr(demeaned["ln_gdp"])
        pc = pd.Series({c: t["ln_x"].corr(t["ln_gdp"]) for c, t in tmp.groupby("CNTR_ID")})

        per_country[col] = pc
        summary.append({
            "feature": col,
            f"between_{year}": round(between, 3),
            "within": round(within, 3),
            "n_countries_corr<0": int((pc < 0).sum()),
            "n_countries_corr<-0.5": int((pc < -0.5).sum()),
        })
    return pd.DataFrame(summary), pd.DataFrame(per_country).round(2)


ntl_gdp = ntl_panel.merge(load_gdp(), on=["CNTR_ID", "year"], how="inner")
print("Rows:", len(ntl_gdp), "| countries:", ntl_gdp["CNTR_ID"].nunique())

corr_summary, corr_by_country = ntl_gdp_correlations(ntl_gdp)
display(corr_summary)
display(corr_by_country.sort_values("ntl_sum_area_weighted"))

Rows: 489 | countries: 35


,feature,between_2024,within,n_countries_corr<0,n_countries_corr<-0.5
0,ntl_sum_area_weighted,0.816,0.343,9,5
1,ntl_sum_cap50,0.824,0.409,4,3
2,ntl_sum_cap100,0.820,0.409,4,4
3,ntl_sum_log1p,0.788,0.435,3,2
4,lit_area_km2_t1,0.806,0.639,3,3
5,lit_area_km2_t2,0.839,0.588,4,3
6,lit_area_km2_t5,0.861,0.451,9,5


,ntl_sum_area_weighted,ntl_sum_cap50,ntl_sum_cap100,ntl_sum_log1p,lit_area_km2_t1,lit_area_km2_t2,lit_area_km2_t5
CH,-0.91,-0.91,-0.91,-0.45,-0.58,-0.91,-0.92
FR,-0.89,-0.89,-0.89,-0.86,-0.87,-0.89,-0.89
NL,-0.89,0.80,0.39,0.90,0.95,0.94,0.88
BE,-0.78,-0.77,-0.77,-0.76,-0.71,-0.77,-0.82
PT,-0.62,-0.43,-0.59,0.74,0.90,0.69,-0.69
ES,-0.06,0.37,0.10,0.66,0.74,0.68,0.27
IS,-0.05,0.10,0.11,0.06,0.83,0.84,0.80
LU,-0.04,0.15,0.09,0.46,0.64,-0.06,-0.29
CZ,-0.00,0.07,0.00,0.33,0.60,0.50,-0.25
SI,0.34,0.35,0.33,0.58,0.68,0.41,-0.37
